# DPO: a good margin can conceal an undesirable policy

Day 18 · Chapter 11. CPU, authored synthetic data, no downloads. Predict before each reference, change one declared variable, and interpret the result. Code is complete so the notebook runs from beginning to end. Completing its execution is material verification, not an assessment of learner mastery.

[Canonical chapter](../../book/chapters/11-direct-preference-optimization.md)


In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / "src" / "dongxi_llms").is_dir():
    if root.parent == root:
        raise RuntimeError("Open this notebook within the Dongxi_LLMs repository")
    root = root.parent
sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.family": "DejaVu Sans", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
blue, orange, green = "#356a9b", "#b97139", "#3d8268"



## Exercise 1: compare the complete response space

Two categorical contexts have three possible answers. Predict what flipping every preference should do. Compare a direct SFT demonstration control, while remembering it has different information.


In [ ]:
from dongxi_llms.dpo_lab import categorical_dpo,tiny_sequence_dpo
results={mode:categorical_dpo(mode) for mode in ("dpo","flipped","sft")}
fig,axes=plt.subplots(1,2,figsize=(11,3.6))
for mode,r in results.items():axes[0].plot(r["history"],label=mode)
axes[0].set(xlabel="optimizer update",ylabel="own training objective");axes[0].legend()
for i,(mode,r) in enumerate(results.items()):
    axes[1].bar(i-.16,r["expected_reward"],width=.32,color=blue)
    axes[1].bar(i+.16,r["reference_kl"],width=.32,color=orange)
axes[1].set_xticks(range(3),list(results));axes[1].set_ylabel("exact reward (blue) / KL nats (orange)")
plt.tight_layout();plt.show()
print({mode:{k:v for k,v in r.items() if k!="history"} for mode,r in results.items()})


![Saved reference plot for 01 dpo controlled comparison, figure 1](../figures/chapter-11/day-18-01_dpo_controlled_comparison-01.png)

Saved reference output from the verified CPU run. Run the preceding cell to regenerate the current experiment; this image does not change when parameters are edited.


**Solution:** flipped labels are a judgment-direction control. SFT is a demonstration control. Their raw loss values measure different targets, so plotting them does not make a lower cross-method loss a quality ranking. Exact enumerated reward and KL provide separate measurements in this known simulator.

## Exercise 2: use an actual decoder

Six symbolic prompts prefer A over B. EOS enters both response scores. Predict whether increasing the relative margin must increase P(A) for an unseen prompt. The following performs real CPU neural optimization and keeps its negative outcomes.


In [ ]:
sequence={mode:tiny_sequence_dpo(mode=mode) for mode in ("dpo","sft")}
fig,axes=plt.subplots(1,2,figsize=(11,3.6))
for mode,r in sequence.items():
    axes[0].plot([h["heldout_A_probability"] for h in r["history"]],label=mode)
axes[0].set(xlabel="optimizer update",ylabel="P(A | held-out symbolic prompt)",ylim=(0,1));axes[0].legend()
axes[1].plot([h["margin"] for h in sequence["dpo"]["history"]],color=blue)
axes[1].set(xlabel="optimizer update",ylabel="mean DPO training pair margin")
plt.tight_layout();plt.show()
print({mode:{k:v for k,v in r.items() if k!="history"} for mode,r in sequence.items()})
assert not sequence["dpo"]["reference_has_gradients"]
assert sequence["dpo"]["reference_self_check"]


![Saved reference plot for 01 dpo controlled comparison, figure 2](../figures/chapter-11/day-18-01_dpo_controlled_comparison-02.png)

Saved reference output from the verified CPU run. Run the preceding cell to regenerate the current experiment; this image does not change when parameters are edited.


**Read the figure:** the left plot is an independent absolute answer-probability metric; the right plot is the trained relative pair diagnostic. They answer different questions. This fixture has one block and twelve symbolic IDs; it does not evaluate English.

## Exercise 3: construct the ratio counterexample

Chosen probability falls from .2 to .1; rejected falls from .1 to .01. Does the reference-relative pair margin improve? Where must the remaining mass go?


In [ ]:
before=torch.tensor([.2,.1,.7],dtype=torch.float64)
after=torch.tensor([.1,.01,.89],dtype=torch.float64)
ratio_before=before[0]/before[1];ratio_after=after[0]/after[1]
relative_margin=.5*torch.log(ratio_after/ratio_before)
print({"chosen_absolute_change":float(after[0]-before[0]),
       "pair_ratio_change":(float(ratio_before),float(ratio_after)),
       "reference_relative_margin":float(relative_margin)})


**Solution:** the pair ratio improves although both scored answers lose mass. A third answer receives the remaining probability. In a real decoder, inspect actual samples, EOS, absolute likelihoods, length, and independent task outputs before naming a cause. The observed synthetic failure does not prove that DPO always fails or that another method is universally better.

**Spark extension:** the canonical lab guide links an optional HF runner with a local SFT checkpoint, fixed reference, template prefix checks, and before/after independent exact generation. That large-model path is proposed, not executed here.
